In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Define the path to the new DMW dataset
dmw_path = '/ceph/MethDev/pbio/andy/JW/section_4/section_4b_chunk_feature_analysis/dmw_methyl_results/combined.tsv'

# Load the TSV file
dmw_df = pd.read_csv(dmw_path, sep='\t')

# Display basic information to verify the load and understand the dimensions
print(f"Loaded DMW shape: {dmw_df.shape}")
display(dmw_df.head())
display(dmw_df.info())

Loaded DMW shape: (38344, 6)


,chunk,meth,total,CellID,genotype,cell_state
0,2:3621500-3621700,1.0,6.0,240614_mct_1_1_P2-4-E5-B7,Col,Phloem
1,2:3626100-3626300,17.0,26.0,240614_mct_1_1_P2-4-E5-B7,Col,Phloem
2,2:9647-9767,6.0,10.0,240614_mct_1_1_P2-4-E5-B7,Col,Phloem
3,3:14203700-14203900,22.0,28.0,240614_mct_1_1_P2-4-E5-B7,Col,Phloem
4,2:3626100-3626300,12.0,15.0,240614_mct_1_1_P2-4-E5-L20,Col,M1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38344 entries, 0 to 38343
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   chunk       38344 non-null  object 
 1   meth        38344 non-null  float64
 2   total       38344 non-null  float64
 3   CellID      38344 non-null  object 
 4   genotype    38344 non-null  object 
 5   cell_state  38344 non-null  object 
dtypes: float64(2), object(4)
memory usage: 1.8+ MB


None

In [26]:
# The most direct, "Pandas" way:
col_unique_count = dmw_df[dmw_df["genotype"] == "Col"]["chunk"].nunique()

print(f"Unique chunks in Col: {col_unique_count}")

Unique chunks in Col: 737


In [9]:
dmw_df["genotype"].unique()

array(['Col', 'rdd', 'T-MET'], dtype=object)

In [16]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.special import logit, expit

# ==========================================
# 1. LOAD AND FILTER DATA
# ==========================================
dmw_path = '/ceph/MethDev/pbio/andy/JW/section_4/section_4b_chunk_feature_analysis/dmw_methyl_results/combined.tsv'
dmw_df = pd.read_csv(dmw_path, sep='\t')

# Filter exactly for the 'Col' genotype
dmw_df_col = dmw_df[dmw_df['genotype'] == 'Col'].copy()
print(f"Filtered DMW shape (Col only): {dmw_df_col.shape}")

# ==========================================
# 2. AGGREGATE AND PIVOT
# ==========================================
# Aggregate counts by chunk and cell_state
agg_df = dmw_df_col.groupby(['chunk', 'cell_state'])[['meth', 'total']].sum().reset_index()

# Pivot to create feature x cluster matrices
cluster_c = agg_df.pivot(index='chunk', columns='cell_state', values='meth').fillna(0)
cluster_m = agg_df.pivot(index='chunk', columns='cell_state', values='total').fillna(0)

# ==========================================
# 3. RENAME COLUMNS TO MATCH YOUR PLOTTING ORDER
# ==========================================
new_cluster_mapping = {
    'M3': '3 Early M',
    'M2': '0 Mid M',       
    'M1': '1 Late M',      
    'Expanding_M': '7 Expanding M',
    'Senescent_M': '10 Senescent M',
    'E1': '5 Early E',    
    'Expanding_E': '13 Expanding E',
    'Abaxial_E': '4 Abaxial E',
    'Adaxial_E': '6 Adaxial E',
    'Senescent_E': '8 Senescent E',
    'Guard_cell': '14 Guard cell',
    'Vasculature': '2 Vasculature',
    'Phloem': '9 Phloem',
    'Myrosinase': '16 Myrosinase',
    'S_phase': '12 S phase',
    'G2M_phase': '15 G2M phase',
    'PP1': '11 PPP' 
}

cluster_c = cluster_c.rename(columns=new_cluster_mapping)
cluster_m = cluster_m.rename(columns=new_cluster_mapping)

# ==========================================
# 4. CALCULATE LOGIT RESIDUALS (NORMALIZATION)
# ==========================================
# Fit Global Offsets (deltas) for each cell state
global_c = cluster_c.sum(axis=0)
global_m = cluster_m.sum(axis=0)
global_p = global_c / global_m

d = logit(np.clip(global_p, 1e-6, 1-1e-6))
deltas = d - np.average(d, weights=global_m)

# Calculate Expected Methylation (p0) using ALL chunks
chunk_c = cluster_c.sum(axis=1)
chunk_m = cluster_m.sum(axis=1)

# Expected values math (no coverage filtering applied)
pbar = np.clip(chunk_c / chunk_m, 1e-6, 1-1e-6)
logit_pbar = logit(pbar).values[:, None]
expected_logit = logit_pbar + deltas.values[None, :]
p0_df = pd.DataFrame(expit(expected_logit), index=cluster_c.index, columns=cluster_c.columns)

# Extract Adjusted Matrix (Residuals)
obs_p = cluster_c / cluster_m.replace(0, np.nan)
obs_p = obs_p.fillna(0) 

adj_meth_matrix = obs_p - p0_df

print(f"Final Unfiltered Meth Residuals matrix shape: {adj_meth_matrix.shape}")

Filtered DMW shape (Col only): (15933, 6)
Final Unfiltered Meth Residuals matrix shape: (737, 20)


In [19]:
cluster_c

cell_state,4 Abaxial E,6 Adaxial E,5 Early E,13 Expanding E,7 Expanding M,15 G2M phase,14 Guard cell,1 Late M,0 Mid M,3 Early M,16 Myrosinase,11 PPP,PP2,9 Phloem,12 S phase,8 Senescent E,10 Senescent M,Unknown,2 Vasculature,Xylem
chunk,,,,,,,,,,,,,,,,,,,,
1:10078700-10079100,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1:10304600-10304900,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1:10307700-10307900,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,12.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1:10327600-10327900,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1:10472600-10473300,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5:9542700-9543200,4.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5:9818924-9819100,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,15.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5:9909800-9910100,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [15]:
dmw_df_col[dmw_df_col["chunk"] == "5:9542700-9543200"]

,chunk,meth,total,CellID,genotype,cell_state
21940,5:9542700-9543200,4.0,5.0,240614_mct_1_4_P1-5-A11-A9,Col,Adaxial_E
29600,5:9542700-9543200,4.0,15.0,240614_mct_1_3_P1-4-K9-I20,Col,Abaxial_E


In [11]:
cluster_m

cell_state,4 Abaxial E,6 Adaxial E,5 Early E,13 Expanding E,7 Expanding M,15 G2M phase,14 Guard cell,1 Late M,0 Mid M,3 Early M,16 Myrosinase,11 PPP,PP2,9 Phloem,12 S phase,8 Senescent E,10 Senescent M,Unknown,2 Vasculature,Xylem
chunk,,,,,,,,,,,,,,,,,,,,
1:10078700-10079100,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1:10304600-10304900,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,31.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1:10307700-10307900,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,30.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1:10327600-10327900,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,15.0,0.0,0.0,0.0,0.0
1:10472600-10473300,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5:9542700-9543200,15.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5:9818924-9819100,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,15.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5:9909800-9910100,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0,0.0,0.0


In [7]:
dmw_df["cell_state"].unique()

array(['Phloem', 'M1', 'Xylem', 'Expanding_E', 'PP2', 'E1', 'Guard_cell',
       'PP1', 'G2M_phase', 'M2', 'Vasculature', 'Expanding_M',
       'Senescent_E', 'Senescent_M', 'M3', 'S_phase', 'Abaxial_E',
       'Myrosinase', 'Adaxial_E', 'Unknown'], dtype=object)

In [27]:
import os

# ==========================================
# EXECUTION: ALL CHUNKS
# ==========================================

# 1. Define where to save the outputs
save_dir = './figures/dmw_heatmaps/'
os.makedirs(save_dir, exist_ok=True)

# 2. Extract ALL chunks from the normalized matrix
all_chunks = adj_meth_matrix.index.tolist()

print(f"Clustering and plotting all {len(all_chunks)} DMWs...")

# 3. Generate the heatmap
plot_dmw_heatmap(
    chunks=all_chunks, 
    plot_cols=cluster_order, # Enforce your custom order with M and E dashed lines
    meth_matrix=adj_meth_matrix,
    title="All Normalized DMWs", 
    save_path=os.path.join(save_dir, "all_dmws_ordered_heatmap.png")
)

Clustering and plotting all 737 DMWs...
✅ Saved: ./figures/dmw_heatmaps/all_dmws_ordered_heatmap.png
